# D2.2 · When the actor is an agent

**Function D — Security Operations → The Incident Responder**  ·  *Security of AI*

---

**Risk.** "Which user" is now the wrong first question.

**Control.** Attribute to agent, authority, delegation chain and prompt.

**This lab.** Attribute an incident to an agent, an authority and a delegation chain.

| | |
|---|---|
| Open-source tooling | Keycloak, OpenSearch |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("D2.2"))

When the actor is an agent, three responder instincts misfire: disable the account, interview the user, and assume one actor.

In [ ]:
from cybercommons import ir, identity
import time

t0 = time.time()
tl = ir.Timeline()
tl.add(t0,      "alice", "alice",        "login")
tl.add(t0 + 10, "alice", "patch-agent",  "write_file", "/etc/app.conf")
tl.add(t0 + 11, "alice", "deploy-agent", "deploy",     "prod")
r = ir.reconstruct(tl)

print("instinct 1 — disable alice's account")
print(f"   agents still running: {r['hidden_actors']}")
print("instinct 2 — interview the user")
print("   alice was asleep. She authorised a task; the agents chose the actions.")
print("instinct 3 — assume one actor")
print(f"   actors in reality: {r['actors_in_reality']}")

The correct first action is revoking the *agent* identity, which is only possible if agents have identities distinct from principals — the A2 control, arriving late.

In [ ]:
reg = identity.Registry()
alice = reg.record(identity.mint("alice"))
patch = reg.record(identity.exchange(alice, "patch-agent", {"repo:write"}))
print(f"revoke patch-agent → {reg.revoke('patch-agent')} token(s) invalidated")
print("alice's own token still valid:", reg.valid(alice))

### Expect

All three instincts are shown to misfire, with `patch-agent` and `deploy-agent` named as hidden actors. Revoking the agent identity invalidates its token while alice's remains valid.

### Your turn

Write the first three steps of your agentic incident runbook. If step one is 'disable the user account', rewrite it.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/D2.2.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*